# Stretching DNA, the worm-like chain

*biophysics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/biophysics/worm_like_chain)

Pull on a DNA molecule with optical tweezers and it resists like an entropic spring that stiffens sharply near full length. The Marko–Siggia worm-like-chain law is compared with a freely jointed chain, and a discrete chain of unit tangents with bending stiffness is simulated by Monte Carlo for a whole batch of forces at once, as one $(B, M, N, 3)$ tensor. The persistence length $P \approx 50$ nm comes back out of both the tangent correlations and a fit to the force–extension curve.

**Vector form**

$$\frac{F P}{k_B T} = \frac{z}{L} + \frac{1}{4(1 - z/L)^2} - \frac14,\qquad \beta E = -K \sum_i \hat{\mathbf t}_i \cdot \hat{\mathbf t}_{i+1} - \frac{F a}{k_B T} \sum_i \hat t_{i,z},\qquad \langle \hat{\mathbf t}_i \cdot \hat{\mathbf t}_{i+n} \rangle = e^{-n a / P}$$

**Index (tensor) form**

$$h_{bmik} = K\,(t_{bm,i-1,k} + t_{bm,i+1,k}) + f_b\,\delta_{kz} \qquad \Delta\beta E_{bmi} = -h_{bmik}\,(t'_{bmik} - t_{bmik}) \qquad \frac{z_{bm}}{L} = \frac1N\, t_{bmiz}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $t_{bmik}$ | `relaxed_chains(B, M, N, K, seed, device)` | (B, M, N, 3) | unit tangent of segment $i$ of chain $m$ pulled by force $b$; chains start as force-free WLCs |
| $F_b,\ f_b$ | `forces(12, device), F * a / KT` | (B,) | $F_0 = 0$ plus 12 forces from 0.02 to 10 pN; $f_b$ is the work $F a$ in units of $k_B T$ |
| $K$ | `stiffness_for(P, a)` | () | bending stiffness $\kappa / k_B T$ chosen so that $\coth K - 1/K = e^{-a/P}$, i.e. $K \approx P/a + \tfrac12$ |
| $h_{bmik}$ | `local_field(t, K, f)` | (B, M, N, 3) | the pull of the two neighbouring tangents plus the force along $z$ |
| $h_{bmik}(t'_{bmik} - t_{bmik})$ | `einsum('bmik,bmik->bmi', h, t_new - t)` | (B, M, N) | repeated $k$ ⇒ summed: energy change of turning one tangent; even and odd segments move in turn |
| $t_{bmik}t_{bm,i+1,k}$ | `einsum('bmik,bmik->bm', t[:, :, :-1], t[:, :, 1:])` | (B, M) | repeated $i, k$ ⇒ summed: total bending energy, used for pivot moves that turn a whole tail |
| $z_{bm}/L$ | `t[..., 2].mean(-1)` | (B, M) | extension along the force over the contour length |
| $\langle t_{ik} t_{i+n,k} \rangle$ | `tangent_correlation(t[0], n_max)` | (n_max+1,) | tangent correlation of the free chain; its decay length is $P$ |

A double-stranded DNA molecule is stiff on short scales but floppy on long ones: the direction of its backbone is remembered over the persistence length $P \approx 50$ nm (about 150 base pairs), then forgotten. Pulling it straight therefore costs entropy, not bond energy. At low force the response is linear, $z/L \approx 2 P F / 3 k_B T$, the same for any polymer with Kuhn length $b = 2P$; at high force the worm-like chain approaches full length only slowly, $1 - z/L \approx \tfrac12 \sqrt{k_B T / F P}$, because it must iron out bending modes of every wavelength, whereas a freely jointed chain of rigid links straightens much faster, $1 - z/L \approx k_B T / F b$. Experiments on λ-phage DNA (Smith, Finzi and Bustamante 1992; Bustamante, Marko, Siggia and Smith 1994) follow the worm-like chain up to about 10 pN, where the double helix starts to stretch elastically, and overstretch near 65 pN; this model has neither. The Marko–Siggia formula is an interpolation between the two limits and differs from the exact WLC by up to about 10 % in force at intermediate extension. The simulation uses segments of $a = 2.5$ nm, so the stiffness is $K \approx 20.5$; each Monte Carlo sweep turns every tangent a little (Metropolis on the local field) and then rotates the tail of every chain about a random axis (a pivot). The first half of the sweeps is equilibration and the second half is averaged. At the lowest forces the extension fluctuates strongly from chain to chain and relaxes slowly, so those points scatter. The fit uses 0.5–4 pN: Marko–Siggia solved for $P$ at each point, averaged in log space (least squares in $\log F$). Above $F \sim K k_B T / a$ the discreteness of the chain shows up as a slightly larger extension than the continuous law. No experimental data are included.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/biophysics/worm_like_chain/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors

N_FORCES = 12  # pulled forces from 0.02 to 10 pN, plus one free chain at F = 0
SHOW = [3, 6, 9, 12]  # forces whose first chain is drawn: about 0.06, 0.3, 2 and 10 pN
FIT_RANGE = (0.5, 4.0)  # pN: well stretched (fast to equilibrate) but far below K k_B T / a (discreteness)


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    P, L = params["persistence"], params["contour_length"]
    a = tensors.SEGMENT
    n_seg = int(round(L / a))
    K = tensors.stiffness_for(P, a)
    F = tensors.forces(N_FORCES, dev)  # (B,) with F[0] = 0
    f = tensors.reduced_forces(F, a)  # (B,)
    M = 2 if fast_mode() else params["chains"]
    sweeps = 40 if fast_mode() else params["sweeps"]
    t0 = tensors.relaxed_chains(len(F), M, n_seg, K, seed=1, device=dev)  # (B, M, N, 3)
    print(f"P = {P:g} nm, L = {L:g} nm → N = {n_seg} segments of a = {a} nm, bending stiffness K = κ/kT = {K:.2f}")
    print(f"{len(F)} forces × {M} chains × {sweeps} Monte Carlo sweeps")

    out = solve.simulate(t0, K, f, sweeps, n_records=60, show=SHOW, seed=0)
    z, z_err = out["z_mean"], out["z_err"]
    z_ms = equation.marko_siggia_extension(F[1:], P)
    z_fjc = equation.fjc_extension(F[1:], 2 * P)
    P_b = equation.persistence_from_force(F[1:], z[1:])  # (B-1,) Marko–Siggia solved for P at each force
    print("   F (pN)   z/L sim         WLC     FJC     P from this point")
    for b in range(1, len(F)):
        print(
            f"{F[b].item():8.3f}   {z[b].item():.3f} ± {z_err[b].item():.3f}   {z_ms[b - 1].item():.3f}   "
            f"{z_fjc[b - 1].item():.3f}   {P_b[b - 1].item():6.1f} nm"
        )
    in_fit = (F[1:] >= FIT_RANGE[0]) & (F[1:] <= FIT_RANGE[1])
    P_fit = torch.exp(torch.log(P_b[in_fit]).mean()).item()  # least squares in log F → geometric mean of P_b
    P_corr = equation.persistence_from_correlation(out["correlation"][: max(2, int(P / a))], a)
    print(
        f"free chain: ⟨t_i·t_i+1⟩ = {out['correlation'][1].item():.4f}  (exact {equation.segment_correlation(K):.4f})"
    )
    print(f"persistence length from the tangent correlation at F = 0: {P_corr:.1f} nm")
    print(f"persistence length from a Marko–Siggia fit, {FIT_RANGE[0]}–{FIT_RANGE[1]} pN: {P_fit:.1f} nm")
    print(f"input persistence length: {P:g} nm")
    return plot.animate(out, F, P, a, SHOW, FIT_RANGE, P_corr, out_dir / "worm_like_chain.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/biophysics/worm_like_chain/equation.py
r"""Physics → tensor translation for a stretched DNA molecule.

Continuous laws (z = extension / contour length, P = persistence length, b = Kuhn length, k_B T = thermal energy):

    worm-like chain (Marko–Siggia):   F P / k_B T = z + 1 / (4 (1 - z)²) - 1/4
    freely jointed chain:             z = coth(F b / k_B T) - k_B T / (F b),      b = 2 P

Discrete worm-like chain: N unit tangents t_{bmik} (force b, chain m, segment i, component k) of length a.
In units of k_B T the energy is bending plus the work done by the force along z:

    βE_{bm} = -K t_{bmik} t_{bm,i+1,k} - f_b t_{bmiz}         summed over i and k;   K = κ / k_B T,  f_b = F_b a / k_B T

Turning one tangent t_i → t'_i changes the energy through its local field h only:

    h_{bmik} = K (t_{bm,i-1,k} + t_{bm,i+1,k}) + f_b δ_{kz}
    ΔβE_{bmi} = -h_{bmik} (t'_{bmik} - t_{bmik})          (k summed)

Neighbouring tangents are correlated as ⟨t_i·t_{i+n}⟩ = (coth K - 1/K)ⁿ = exp(-n a / P).
"""

import math

import torch

KT = 4.11  # thermal energy k_B T at room temperature, pN·nm


def marko_siggia_force(z: torch.Tensor, P: float) -> torch.Tensor:
    """F(z) = k_B T / P · (z + 1/(4(1-z)²) - 1/4) in pN, for relative extension 0 ≤ z < 1."""
    return KT / P * (z + 0.25 / (1 - z) ** 2 - 0.25)


def marko_siggia_extension(F: torch.Tensor, P: float, iterations: int = 60) -> torch.Tensor:
    """Invert Marko–Siggia for z(F) by bisection (the force grows monotonically with z), elementwise on any shape."""
    lo, hi = torch.zeros_like(F), torch.ones_like(F)
    for _ in range(iterations):
        mid = 0.5 * (lo + hi)
        too_far = marko_siggia_force(mid, P) > F
        hi = torch.where(too_far, mid, hi)
        lo = torch.where(too_far, lo, mid)
    return 0.5 * (lo + hi)


def fjc_extension(F: torch.Tensor, b: float) -> torch.Tensor:
    """Freely jointed chain: z = L(F b / k_B T) with the Langevin function L(x) = coth x - 1/x."""
    x = F * b / KT
    return 1 / torch.tanh(x) - 1 / x


def bending_energy(t: torch.Tensor, K: float, f: torch.Tensor) -> torch.Tensor:
    """βE_bm = -K t_{bmik} t_{bm,i+1,k} - f_b t_{bmiz}, repeated i and k summed.   t: (B, M, N, 3), f: (B,) → (B, M)."""
    bend = torch.einsum("bmik,bmik->bm", t[:, :, :-1], t[:, :, 1:])  # Σ_i t_i·t_{i+1}
    pull = t[..., 2].sum(-1)  # Σ_i t_iz
    return -K * bend - f[:, None] * pull


def local_field(t: torch.Tensor, K: float, f: torch.Tensor) -> torch.Tensor:
    """h_{bmik} = K (t_{i-1} + t_{i+1})_k + f_b δ_{kz}; the two free ends have only one neighbour.   (B, M, N, 3)."""
    pad = torch.nn.functional.pad(t, (0, 0, 1, 1))  # zero tangents beyond both ends: (B, M, N+2, 3)
    h = K * (pad[:, :, :-2] + pad[:, :, 2:])
    h[..., 2] += f[:, None, None]
    return h


def local_energy_change(h: torch.Tensor, t: torch.Tensor, t_new: torch.Tensor) -> torch.Tensor:
    """ΔβE_{bmi} = -h_{bmik} (t'_{bmik} - t_{bmik}), repeated k summed.   (B, M, N)."""
    return -torch.einsum("bmik,bmik->bmi", h, t_new - t)


def relative_extension(t: torch.Tensor) -> torch.Tensor:
    """z / L = (1/N) Σ_i t_{bmiz}: end-to-end distance along the force over the contour length.   (B, M)."""
    return t[..., 2].mean(-1)


def tangent_correlation(t: torch.Tensor, max_lag: int) -> torch.Tensor:
    """⟨t_i · t_{i+n}⟩ averaged over i and all chains in t (..., N, 3), for n = 0 … max_lag.   (max_lag+1,)."""
    n = t.shape[-2]
    return torch.stack(
        [torch.einsum("...ik,...ik->...i", t[..., : n - s, :], t[..., s:, :]).mean() for s in range(max_lag + 1)]
    )


def segment_correlation(K: float) -> float:
    """⟨t_i · t_{i+1}⟩ = coth K - 1/K for a discrete chain with bending stiffness K (the Langevin function)."""
    return 1 / math.tanh(K) - 1 / K


def persistence_from_stiffness(K: float, a: float) -> float:
    """P = -a / ln(coth K - 1/K), from ⟨t_i·t_{i+n}⟩ = exp(-n a / P)."""
    return -a / math.log(segment_correlation(K))


def persistence_from_correlation(c: torch.Tensor, a: float) -> float:
    """Least-squares slope of ln⟨t_i·t_{i+n}⟩ = -n a / P through the origin:  P = -a Σ n² / Σ n ln c_n.

    c: (n_max+1,) correlations for lags n = 0 … n_max.
    """
    n = torch.arange(len(c), device=c.device, dtype=c.dtype)
    return (-a * (n**2).sum() / (n * torch.log(c)).sum()).item()


def persistence_from_force(F: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    """Solve Marko–Siggia for P at each measured point: P_b = k_B T (z + 1/(4(1-z)²) - 1/4) / F_b.   (B,)"""
    return KT * (z + 0.25 / (1 - z) ** 2 - 0.25) / F


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/biophysics/worm_like_chain/tensors.py
"""Build the concrete tensors: the applied forces, the bending stiffness that gives the wanted persistence length,
and the starting chains. Units: lengths in nm, forces in pN, energies in k_B T (k_B T = 4.11 pN·nm).
"""

import math

import torch

from .equation import KT, persistence_from_stiffness

SEGMENT = 2.5  # segment length a in nm: much shorter than P, so the discrete chain behaves like a continuous WLC


def forces(n: int, device: torch.device) -> torch.Tensor:
    """F_b: F_0 = 0 (a free chain, used for the tangent correlation), then n forces spread evenly on a log scale
    from 0.02 pN to 10 pN, like an optical- or magnetic-tweezers run.   Shape (B,) = (n + 1,)."""
    pulled = torch.logspace(math.log10(0.02), 1.0, n, device=device)
    return torch.cat([torch.zeros(1, device=device), pulled])


def reduced_forces(F: torch.Tensor, a: float) -> torch.Tensor:
    """f_b = F_b a / k_B T: the work done by the force on one segment that points along it.   (B,)"""
    return F * a / KT


def stiffness_for(P: float, a: float) -> float:
    """Bending stiffness K = κ / k_B T for which the discrete chain has persistence length P exactly.

    P(K) = -a / ln(coth K - 1/K) increases with K, so a bisection finds it; K ≈ P/a + 1/2 for P ≫ a.
    """
    lo, hi = 1e-3, 10 * P / a + 10
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if persistence_from_stiffness(mid, a) < P:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def relaxed_chains(n_forces: int, n_chains: int, n_segments: int, K: float, seed: int, device: torch.device):
    """t_{bmik}: chains drawn exactly from the force-free worm-like chain, so the run starts close to equilibrium.

    Without force each tangent depends only on the one before: the angle θ to it follows p(cos θ) ∝ exp(K cos θ),
    sampled by inverting its cumulative distribution, in a random direction around it.   (B, M, N, 3)
    """
    gen = torch.Generator().manual_seed(seed)
    shape = (n_forces, n_chains)
    t = torch.nn.functional.normalize(torch.randn(*shape, 3, generator=gen), dim=-1)  # first tangent: random
    chain = [t]
    for _ in range(n_segments - 1):
        u = torch.rand(shape, generator=gen)
        cos = 1 + torch.log(u + (1 - u) * math.exp(-2 * K)) / K  # (B, M), cos θ between t_i and t_{i+1}
        g = torch.randn(*shape, 3, generator=gen)
        side = torch.nn.functional.normalize(g - (g * t).sum(-1, keepdim=True) * t, dim=-1)  # random ⟂ t_i
        t = cos[..., None] * t + torch.sqrt((1 - cos**2).clamp(min=0))[..., None] * side
        t = torch.nn.functional.normalize(t, dim=-1)  # remove round-off so |t| stays exactly 1
        chain.append(t)
    return torch.stack(chain, dim=2).to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/biophysics/worm_like_chain/solve.py
"""Metropolis Monte Carlo for every force and every chain at once: local tangent rotations plus pivot moves.

One sweep = update the even segments, then the odd ones (a segment's energy change depends only on its two
neighbours, so all even segments can move together), then one pivot move per chain that turns the whole tail of
the chain at once, which relaxes the long, floppy shapes at low force much faster than local moves alone.
"""

import torch

from problems._common.progress import steps

from .equation import bending_energy, local_energy_change, local_field, relative_extension, tangent_correlation


def local_moves(t: torch.Tensor, K: float, f: torch.Tensor, step: float, gen: torch.Generator) -> torch.Tensor:
    """Try t_i → normalise(t_i + step·ξ) on the even, then the odd segments of every chain.   t: (B, M, N, 3)."""
    n = t.shape[2]
    for parity in (0, 1):
        noise = torch.randn(t.shape, generator=gen).to(t.device)
        t_new = torch.nn.functional.normalize(t + step * noise, dim=-1)  # a small random turn, still a unit vector
        dE = local_energy_change(local_field(t, K, f), t, t_new)  # (B, M, N)
        u = torch.rand(dE.shape, generator=gen).to(t.device)
        on_parity = (torch.arange(n, device=t.device) % 2 == parity)[None, None, :]
        accept = (u < torch.exp(-dE)) & on_parity
        t = torch.where(accept[..., None], t_new, t)
    return t


def rotation_matrices(axis: torch.Tensor, angle: torch.Tensor) -> torch.Tensor:
    """Rodrigues: R = I + sin θ [u]× + (1 - cos θ) [u]×², for unit axes u (..., 3) and angles θ (...).   (..., 3, 3)"""
    ux, uy, uz = axis.unbind(-1)
    zero = torch.zeros_like(ux)
    cross = torch.stack([zero, -uz, uy, uz, zero, -ux, -uy, ux, zero], dim=-1).reshape(*axis.shape, 3)
    eye = torch.eye(3, device=axis.device).expand_as(cross)
    s, c = torch.sin(angle)[..., None, None], torch.cos(angle)[..., None, None]
    return eye + s * cross + (1 - c) * cross @ cross


def pivot_move(t: torch.Tensor, K: float, f: torch.Tensor, max_angle: float, gen: torch.Generator) -> torch.Tensor:
    """Rotate segments i, i+1, …, N-1 of each chain about a random axis: t'_j = R t_j for j ≥ i."""
    B, M, n, _ = t.shape
    dev = t.device
    axis = torch.nn.functional.normalize(torch.randn(B, M, 3, generator=gen), dim=-1).to(dev)
    angle = (max_angle * (2 * torch.rand(B, M, generator=gen) - 1)).to(dev)
    i = torch.randint(0, n, (B, M, 1), generator=gen).to(dev)
    R = rotation_matrices(axis, angle)  # (B, M, 3, 3)
    turned = torch.einsum("bmkl,bmil->bmik", R, t)  # every segment rotated ...
    t_new = torch.where((torch.arange(n, device=dev) >= i)[..., None], turned, t)  # ... but only the tail kept
    dE = bending_energy(t_new, K, f) - bending_energy(t, K, f)  # (B, M)
    accept = torch.rand(B, M, generator=gen).to(dev) < torch.exp(-dE)
    return torch.where(accept[:, :, None, None], t_new, t)


def simulate(
    t0: torch.Tensor, K: float, f: torch.Tensor, sweeps: int, n_records: int, show: list[int], seed: int = 0
) -> dict:
    """Run the Monte Carlo; the first half of the sweeps equilibrates and the second half is measured.

    show: force indices whose chain m = 0 is stored for the animation.
    """
    gen = torch.Generator().manual_seed(seed)
    step = min(0.5, 1.5 / K**0.5)  # turn size matched to the thermal wobble of one segment, ~1/√(2K)
    every = max(1, sweeps // n_records)
    t = t0.clone()
    z = torch.empty(sweeps + 1, *t.shape[:2], device=t.device)  # (S+1, B, M) extension after every sweep
    z[0] = relative_extension(t)
    snapshots, record_sweeps = [t[show, 0].clone()], [0]
    max_lag = min(60, t.shape[2] - 1)
    corr, n_corr = torch.zeros(max_lag + 1, device=t.device), 0

    for s in steps(sweeps, "MC sweeps"):
        t = local_moves(t, K, f, step, gen)
        t = pivot_move(t, K, f, 0.6, gen)
        z[s + 1] = relative_extension(t)
        if (s + 1) % every == 0 or s == sweeps - 1:
            snapshots.append(t[show, 0].clone())
            record_sweeps.append(s + 1)
        if s >= sweeps // 2 and s % 10 == 0:  # tangent correlations at the lowest force, measured half
            corr += tangent_correlation(t[0], max_lag)
            n_corr += 1

    return {
        "t": t,  # (B, M, N, 3) final tangents
        "z": z,  # (S+1, B, M) relative extension history
        "z_mean": z[sweeps // 2 + 1 :].mean(dim=(0, 2)),  # (B,) measured ⟨z⟩/L at every force
        "z_err": z[sweeps // 2 + 1 :].mean(0).std(-1) / t.shape[1] ** 0.5,  # (B,) error from chain-to-chain spread
        "snapshots": torch.stack(snapshots),  # (K, len(show), N, 3)
        "record_sweeps": torch.tensor(record_sweeps),  # (K,)
        "correlation": corr / max(n_corr, 1),  # (max_lag+1,) ⟨t_i·t_{i+n}⟩ at the lowest force
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/biophysics/worm_like_chain/plot.py
"""Animate the force–extension curve filling in as the Monte Carlo runs, next to chains pulled by four forces."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import KT, fjc_extension, marko_siggia_extension

COLORS = ["#2563eb", "#16a34a", "#ea580c", "#dc2626"]


def animate(
    out: dict,
    F: torch.Tensor,
    P: float,
    a: float,
    show: list[int],
    fit_range: tuple[float, float],
    P_corr: float,
    path: Path,
) -> Path:
    forces = F.cpu().numpy()
    z_chain = out["z"].mean(-1).cpu().numpy()  # (S+1, B) extension averaged over chains, after every sweep
    rec = out["record_sweeps"].numpy()
    # running average over the second half of the sweeps done so far (the first half counts as equilibration)
    cum = np.concatenate([np.zeros((1, z_chain.shape[1])), np.cumsum(z_chain, axis=0)])
    start = rec // 2
    z_run = (cum[rec + 1] - cum[start]) / (rec + 1 - start)[:, None]  # (K, B)
    chains = (a * torch.cumsum(out["snapshots"], dim=2)).cpu().numpy()  # (K, 4, N, 3) bead positions in nm
    in_fit = (forces >= fit_range[0]) & (forces <= fit_range[1])

    F_line = torch.logspace(-2.2, 1.4, 300, dtype=torch.float64)
    z_ms = marko_siggia_extension(F_line, P).numpy()
    z_fjc = fjc_extension(F_line, 2 * P).numpy()

    fig, (ax_f, ax_c) = plt.subplots(1, 2, figsize=(9.5, 4.2), gridspec_kw={"width_ratios": [1.15, 1]})

    # --- force against extension ---
    ax_f.set_yscale("log")
    ax_f.minorticks_off()
    ax_f.set_xlim(0, 1)
    ax_f.set_ylim(F_line[0].item(), F_line[-1].item())
    ax_f.set_yticks([0.01, 0.1, 1, 10], ["0.01", "0.1", "1", "10"])
    ax_f.axhspan(*fit_range, color="#fef3c7", zorder=0, label="fit range")
    ax_f.plot(z_ms, F_line.numpy(), color="#2563eb", lw=1.8, label=f"worm-like chain, P = {P:g} nm")
    ax_f.plot(z_fjc, F_line.numpy(), "--", color="#6b7280", lw=1.3, label=f"freely jointed, b = 2P = {2 * P:g} nm")
    (dots,) = ax_f.plot([], [], "o", color="#ea580c", ms=5, mec="#9a3412", label="Monte Carlo, discrete WLC")
    ax_f.set_xlabel("relative extension  z / L")
    ax_f.set_ylabel("force F (pN)")
    ax_f.legend(loc="upper left", fontsize=7.5)
    readout = ax_f.text(0.03, 0.62, "", transform=ax_f.transAxes, fontsize=8.5, va="top")
    title = ax_f.set_title("force–extension after 0 sweeps", fontsize=10)  # real text so tight_layout leaves room

    # --- chains pulled along z, side view ---
    gap = 0.45 * chains.shape[2] * a  # horizontal spacing of the four chains
    ax_c.set_aspect("equal")
    ax_c.set_xlim(-0.6 * gap, 3.6 * gap)
    ax_c.set_ylim(-0.25 * gap, chains.shape[2] * a * 1.15)
    ax_c.set_xticks([])
    ax_c.set_ylabel("z (nm), along the force")
    ax_c.grid(False)
    lines, arrows = [], []
    arrow_len = 0.12 * chains.shape[2] * a
    for c, b in enumerate(show):
        arrows.append(  # the pulling force, drawn at the free end of the chain
            ax_c.annotate("", xy=(0, 0), xytext=(0, 0), arrowprops={"arrowstyle": "-|>", "color": "#dc2626", "lw": 1.5})
        )
        ax_c.text(c * gap, -0.18 * gap, f"{forces[b]:.2g} pN", ha="center", fontsize=8, color=COLORS[c])
        lines.append(ax_c.plot([], [], lw=1.2, color=COLORS[c])[0])
    ax_c.axhline(0, color="#374151", lw=2)
    ax_c.set_title("one chain at each of four forces", fontsize=10)

    def update(k: int):
        dots.set_data(z_run[k, 1:], forces[1:])
        P_b = KT * (z_run[k, 1:] + 0.25 / (1 - z_run[k, 1:]) ** 2 - 0.25) / forces[1:]
        P_in = P_b[in_fit[1:]]
        fit = f"{np.exp(np.log(P_in).mean()):.0f} nm" if (P_in > 0).all() else "…"
        readout.set_text(f"fit: P = {fit}\ntangent correlation: P = {P_corr:.0f} nm")
        title.set_text(f"force–extension after {rec[k]} sweeps")
        for c in range(len(show)):
            r = chains[k, c] - chains[k, c, :1]  # anchor the first bead at the surface
            lines[c].set_data(r[:, 0] + c * gap, r[:, 2])
            end = (r[-1, 0] + c * gap, r[-1, 2])
            arrows[c].xy = (end[0], end[1] + arrow_len)
            arrows[c].set_position(end)

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(rec), 60), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'persistence': 50.0,
    'contour_length': 1000.0,
    'chains': 12,
    'sweeps': 1200
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.biophysics.worm_like_chain")]:
    del sys.modules[name]
main = importlib.import_module("problems.biophysics.worm_like_chain.main")
problem_dir = Path("problems/biophysics/worm_like_chain")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
